In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV,train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
import numpy as np
import pandas as pd


In [6]:
data=pd.read_csv(r"C:\Users\abody\Desktop\Final Project Sprints\Heart_Disease_Project\data\Cleaned")
df=pd.DataFrame(data) 

In [7]:
X = df.drop(columns=["Target"])
y = df["Target"]
X_train,X_test,y_train,y_test=train_test_split(X,y,train_size=0.8,random_state=32)

In [11]:
log_reg = LogisticRegression(max_iter=5000, random_state=42)

param_grid = [
    # Valid L2 penalties
    {'penalty': ['l2'], 'solver': ['lbfgs', 'saga'], 'C': [0.01, 0.1, 1, 10, 100]},
    
    # Valid L1 penalties (only saga or liblinear)
    {'penalty': ['l1'], 'solver': ['saga', 'liblinear'], 'C': [0.01, 0.1, 1, 10, 100]},
    
    # Elasticnet (requires l1_ratio)
    {'penalty': ['elasticnet'], 'solver': ['saga'], 'C': [0.01, 0.1, 1, 10, 100], 'l1_ratio': [0.1, 0.5, 0.9]}
]

grid_search = GridSearchCV(
    log_reg,
    param_grid,
    cv=5,
    scoring='f1_weighted',
    n_jobs=-1,
    error_score='raise'  # <- this will show errors directly if something is invalid
)

grid_search.fit(X_train, y_train)

print("Best Params:", grid_search.best_params_)
print("Best F1 Score:", grid_search.best_score_)


Best Params: {'C': 1, 'penalty': 'l2', 'solver': 'lbfgs'}
Best F1 Score: 0.5694996934723677


In [13]:

rf = RandomForestClassifier(random_state=42)

# Define parameter space
param_dist = {
    'n_estimators': np.arange(50, 300, 50),
    'max_depth': [None, 5, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'bootstrap': [True, False]
}

random_search = RandomizedSearchCV(
    rf, 
    param_distributions=param_dist,
    n_iter=30, 
    cv=5, 
    scoring='f1_weighted',
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train, y_train)

print("Best Parameters (RandomForest):", random_search.best_params_)
print("Best Score (F1):", random_search.best_score_)
best_rf = random_search.best_estimator_


Best Parameters (RandomForest): {'n_estimators': np.int64(150), 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_depth': 10, 'bootstrap': False}
Best Score (F1): 0.5405448216086514


In [ ]:
tuned_results = pd.DataFrame([
    {"Model": "Logistic Regression (Tuned)", "Best Params": grid_search.best_params_, "Best F1": grid_search.best_score_},
    {"Model": "Random Forest (Tuned)", "Best Params": random_search.best_params_, "Best F1": random_search.best_score_}
])

                         Model  \
0  Logistic Regression (Tuned)   
1        Random Forest (Tuned)   

                                         Best Params   Best F1  
0       {'C': 1, 'penalty': 'l2', 'solver': 'lbfgs'}  0.569500  
1  {'n_estimators': 150, 'min_samples_split': 2, ...  0.540545  


In [ ]:
tuned_results

,Model,Best Params,Best F1
0,Logistic Regression (Tuned),"{'C': 1, 'penalty': 'l2', 'solver': 'lbfgs'}",0.569500
1,Random Forest (Tuned),"{'n_estimators': 150, 'min_samples_split': 2, ...",0.540545


In [ ]:
import joblib
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pipeline = Pipeline([
    ("scaler", StandardScaler()),  # scale data the same way every time
    ("model", LogisticRegression(**grid_search.best_params_))  # best tuned model
])

# 2. Fit the pipeline on your training data
pipeline.fit(X_train, y_train)

# 3. Save pipeline as .pkl file
joblib.dump(pipeline, r"C:\Users\abody\Desktop\Final Project Sprints\Heart_Disease_Project\models\best_heart_disease_model.pkl")

print("Model pipeline saved as best_heart_disease_model.pkl")


✅ Model pipeline saved as best_heart_disease_model.pkl
